# Cvičení: Testování hypotéz - Funguje nová sleva?


https://pslib.sharepoint.com/sites/studium/it/uin/SitePages/Ex-DataTesting.aspx?TeamsCID=80a85cfe-9ea1-404a-bb6e-ce3d008708ac

In [14]:
from cairo import Path
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from google.colab import files
import io

df = pd.read_csv("/content/data/Testing_Hypothetis.csv")

# 2) Data: A = staré, B = nové
n_A, x_A = df.loc[0, "Počet návštěvníků"], df.loc[0, "Počet nákupů"]
n_B, x_B = df.loc[1, "Počet návštěvníků"], df.loc[1, "Počet nákupů"]
# (alternativně natvrdo: n_A = n_B = 5000; x_A, x_B = 210, 245)

p_A, p_B = x_A / n_A, x_B / n_B
print(f"A: {p_A:.2%}, B: {p_B:.2%}, rozdíl: {(p_B - p_A)*100:.2f} p. b.")

# 3) Dvouvýběrový z-test podílů
# H0: p_B = p_A, H1 (jednostranná): p_B > p_A
z, p_one = proportions_ztest([x_B, x_A], [n_B, n_A], alternative="larger")
_, p_two = proportions_ztest([x_B, x_A], [n_B, n_A], alternative="two-sided")
print(f"z = {z:.3f}")
print(f"p (jednostranný) = {p_one:.4f}")
print(f"p (dvoustranný)  = {p_two:.4f}")
print("Zamítáme H0 při α = 0,05 (jednostranně)?", p_one < 0.05)
print("Zamítáme H0 při α = 0,05 (dvoustranně)?", p_two < 0.05)

# 4) 95% interval spolehlivosti pro rozdíl podílů
se = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)
diff = p_B - p_A
lo, hi = diff - 1.96*se, diff + 1.96*se
print(f"95% CI rozdílu: ({lo*100:.2f}; {hi*100:.2f}) p. b.")

# 5) Věcný dopad v Kč (upravte vlastní odhady)
navstevniku_rocne = 500_000
marze_na_objednavku = 200     # Kč
extra = diff * navstevniku_rocne
print(f"Odhad: {extra:.0f} nákupů navíc ročně ≈ {extra*marze_na_objednavku:,.0f} Kč")
print(f"Při dolní/horní mezi CI: {lo*navstevniku_rocne*marze_na_objednavku:,.0f} / "
      f"{hi*navstevniku_rocne*marze_na_objednavku:,.0f} Kč")

# 6) Co se stane s p-hodnotou při větším vzorku (stejné podíly 4,2 % vs. 4,9 %)
print("\nVliv velikosti vzorku (na každou skupinu):")
for n in [500, 1_000, 5_000, 50_000, 500_000]:
    z_n, p_n = proportions_ztest([0.049*n, 0.042*n], [n, n], alternative="larger")
    print(f"n = {n:>7}: z = {z_n:6.2f}, p = {p_n:.3g}")

A: 4.20%, B: 4.90%, rozdíl: 0.70 p. b.
z = 1.679
p (jednostranný) = 0.0465
p (dvoustranný)  = 0.0931
Zamítáme H0 při α = 0,05 (jednostranně)? True
Zamítáme H0 při α = 0,05 (dvoustranně)? False
95% CI rozdílu: (-0.12; 1.52) p. b.
Odhad: 3500 nákupů navíc ročně ≈ 700,000 Kč
Při dolní/horní mezi CI: -116,805 / 1,516,805 Kč

Vliv velikosti vzorku (na každou skupinu):
n =     500: z =   0.53, p = 0.298
n =    1000: z =   0.75, p = 0.226
n =    5000: z =   1.68, p = 0.0465
n =   50000: z =   5.31, p = 5.45e-08
n =  500000: z =  16.79, p = 1.33e-63
